In [ ]:
import time
import numpy as np
from math import sqrt
from numba import cuda, float64, jit, vectorize
import warnings

# Clear all warnings
warnings.filterwarnings("ignore")

In [ ]:
N = 150000000
h = 1.0

@vectorize([float64(float64, float64)])
def count_func_on_CPU(N, h):
    i = 1.0
    result = []
    while i <= N + 1:
        result.append(sqrt(i - 1) + 1 / (i + 3))
        i += h
    return np.sum(result)

start = time.time()
count_func_on_CPU(N, h)
end = time.time()

print(f"Time : {end - start} sec")

Time : 84.60946011543274 sec


In [ ]:
@cuda.jit(device=True)
def f(i):
    return sqrt(i - 1) + 1 / (i + 3)

@cuda.jit
def count_func_on_GPU(result, N, k):
    i = cuda.grid(1)
    h = N / k
    if i < result.size:
        x = (i + 1) * h
        result[i] = f(x)

result = np.zeros(N // 2)

start = time.time()
count_func_on_GPU[(result.size + (128 - 1)) // 128, 128](result, N, N // 2)
end = time.time()

print(f"Time : {end - start} sec")

Time : 1.746828556060791 sec


In [ ]:
import random
import string

In [ ]:
stringi = "Lorem ipsum dolor sit amet, consectetur adipiscing Lorem. Ut dapibus pretium dui, vitae bibendum leo ullamcorper sit amet. Curabitur eu libero tempus dui fringilla gravida non vel tellus. Nulla facilisi. Sed enim diam, convallis id convallis in, varius eget ex. Nulla eget scelerisque ipsum. Proin fermentum, nisi ac gravida tincidunt, neque nibh rutrum odio, non viverra massa tortor non augue. Vestibulum ante ipsum primis in faucibus orci luctus et ultrices posuere cubilia Lorem; Nam dui sem, dictum iaculis urna vel, commodo malesuada nunc. Duis velit lorem, fringilla vel ultrices ut, suscipit eget neque. Nunc convallis ac elit laoreet gravida. Fusce id mauris et nisl viverra cursus finibus id elit. Proin et augue felis. Vivamus elementum elit sodales enim fermentum, sed elementum lorem efficitur. Aenean egestas pretium tellus non ornare. Vestibulum ante ipsum primis in faucibus orci luctus et ultrices posuere Lorem curae; Mauris viverra mi ultrices ultricies efficitur. Duis dignissim orci in nulla egestas bibendum. Mauris iaculis posuere tellus a accumsan. Maecenas condimentum rutrum turpis eu blandit. Quisque viverra vitae ligula non blandit. Suspendisse sodales vulputate sapien ut interdum. Sed rhoncus neque ac Lorem rutrum, ac ultrices ante venenatis. Etiam nisi ipsum, fringilla et pellentesque id, commodo a ligula. Nam rutrum aliquet malesuada. Donec ac nisi ultricies, fermentum augue nec, volutpat purus. Aenean sed lacinia neque. Praesent id velit ante. Pellentesque et tortor non erat volutpat varius. Sed malesuada ut orci in molestie. Pellentesque Lorem tincidunt sapien. Nulla enim nisi, pharetra iaculis ligula sit amet, blandit mattis nisl. Vivamus at malesuada diam, varius mattis erat. Etiam ac bibendum leo. Quisque eu lacus eget ligula imperdiet egestas non vel purus. Pellentesque facilisis porttitor magna, in dignissim massa mattis vitae. Praesent non velit quis quam gravida Lorem nec in diam. Nulla at consectetur nunc. Nullam dui dolor, venenatis id magna a, semper placerat mauris. Donec est purus, imperdiet sit amet mauris in, pellentesque pharetra nisl. Duis nec efficitur urna. Nunc convallis tristique nisi, vel mollis nisl Lorem et. Vestibulum ante ipsum primis in faucibus orci luctus et ultrices posuere cubilia curae; Etiam convallis massa vulputate ipsum malesuada auctor. Integer pellentesque sapien id facilisis tempor. Etiam blandit sodales scelerisque. Vestibulum ac metus tristique, suscipit ante nec, condimentum leo. Pellentesque vel lectus nec nunc pulvinar elementum et a augue. Curabitur sollicitudin lorem sed tortor suscipit, ac auctor mi dictum. In ut nunc elit. Vivamus diam libero, egestas in augue vitae, tempor finibus mi. Donec tincidunt blandit mauris, a volutpat orci hendrerit faucibus. Suspendisse finibus congue dui, in scelerisque ante ultricies ornare. Suspendisse finibus pulvinar nulla quis lacinia. Nam molestie tempus elit. Phasellus vehicula orci ut luctus lacinia."

def generate_random_string(n):
    return ''.join(random.choices(string.ascii_letters + string.digits, k=n))

# stringi += generate_random_string(1000000)


word = "Lorem"


def count_word_on_CPU():
  counter = 0
  for i in range(len(stringi) - len(word) - 1):
    if stringi[i:i+len(word)] == word:
      counter += 1
  return counter

start = time.time()
count_word_on_CPU()
end = time.time()

print(f"Time : {end - start} sec")



Time : 0.0012733936309814453 sec


In [ ]:
@cuda.jit
def count_hello_gpu(string_array, result):
    idx = cuda.grid(1)
    if idx < string_array.size:
        if string_array[idx] == "Hello":
            cuda.atomic.add(result, 0, 1)

def count_hello(string):
    # Split the string into words
    words = string.split()

    # Convert to numpy array
    string_array = np.array(words)

    # Allocate memory on the device
    d_string_array = cuda.to_device(string_array)
    d_result = cuda.to_device(np.array([0], dtype=np.int32))

    # Define block and grid dimensions
    threads_per_block = 128
    blocks_per_grid = (string_array.size + threads_per_block - 1) // threads_per_block

    # Launch the kernel
    start = time.time()
    count_hello_gpu[blocks_per_grid, threads_per_block](d_string_array, d_result)
    end = time.time()
    print(f"Time : {end - start} sec")

    # Copy the result back to host
    result = d_result.copy_to_host()[0]
    return result

if __name__ == '__main__':
    stringg ="Lorem ipsum dolor sit amet, consectetur adipiscing Lorem. Ut dapibus pretium dui, vitae bibendum leo ullamcorper sit amet. Curabitur eu libero tempus dui fringilla gravida non vel tellus. Nulla facilisi. Sed enim diam, convallis id convallis in, varius eget ex. Nulla eget scelerisque ipsum. Proin fermentum, nisi ac gravida tincidunt, neque nibh rutrum odio, non viverra massa tortor non augue. Vestibulum ante ipsum primis in faucibus orci luctus et ultrices posuere cubilia Lorem; Nam dui sem, dictum iaculis urna vel, commodo malesuada nunc. Duis velit lorem, fringilla vel ultrices ut, suscipit eget neque. Nunc convallis ac elit laoreet gravida. Fusce id mauris et nisl viverra cursus finibus id elit. Proin et augue felis. Vivamus elementum elit sodales enim fermentum, sed elementum lorem efficitur. Aenean egestas pretium tellus non ornare. Vestibulum ante ipsum primis in faucibus orci luctus et ultrices posuere Lorem curae; Mauris viverra mi ultrices ultricies efficitur. Duis dignissim orci in nulla egestas bibendum. Mauris iaculis posuere tellus a accumsan. Maecenas condimentum rutrum turpis eu blandit. Quisque viverra vitae ligula non blandit. Suspendisse sodales vulputate sapien ut interdum. Sed rhoncus neque ac Lorem rutrum, ac ultrices ante venenatis. Etiam nisi ipsum, fringilla et pellentesque id, commodo a ligula. Nam rutrum aliquet malesuada. Donec ac nisi ultricies, fermentum augue nec, volutpat purus. Aenean sed lacinia neque. Praesent id velit ante. Pellentesque et tortor non erat volutpat varius. Sed malesuada ut orci in molestie. Pellentesque Lorem tincidunt sapien. Nulla enim nisi, pharetra iaculis ligula sit amet, blandit mattis nisl. Vivamus at malesuada diam, varius mattis erat. Etiam ac bibendum leo. Quisque eu lacus eget ligula imperdiet egestas non vel purus. Pellentesque facilisis porttitor magna, in dignissim massa mattis vitae. Praesent non velit quis quam gravida Lorem nec in diam. Nulla at consectetur nunc. Nullam dui dolor, venenatis id magna a, semper placerat mauris. Donec est purus, imperdiet sit amet mauris in, pellentesque pharetra nisl. Duis nec efficitur urna. Nunc convallis tristique nisi, vel mollis nisl Lorem et. Vestibulum ante ipsum primis in faucibus orci luctus et ultrices posuere cubilia curae; Etiam convallis massa vulputate ipsum malesuada auctor. Integer pellentesque sapien id facilisis tempor. Etiam blandit sodales scelerisque. Vestibulum ac metus tristique, suscipit ante nec, condimentum leo. Pellentesque vel lectus nec nunc pulvinar elementum et a augue. Curabitur sollicitudin lorem sed tortor suscipit, ac auctor mi dictum. In ut nunc elit. Vivamus diam libero, egestas in augue vitae, tempor finibus mi. Donec tincidunt blandit mauris, a volutpat orci hendrerit faucibus. Suspendisse finibus congue dui, in scelerisque ante ultricies ornare. Suspendisse finibus pulvinar nulla quis lacinia. Nam molestie tempus elit. Phasellus vehicula orci ut luctus lacinia."


    count_hello(stringg)



Time : 26.84172749519348 sec


In [ ]:
len(stringg)

2951